# Projet MLOps - Mental Health Condition Prediction
## Mise en production et déploiement continu

**Dataset**: `mental_health_dataset.csv` — prédiction binaire de la condition de santé mentale (Yes/No)

**Pipeline**:
1. Préparation des données (encodage, normalisation, PCA embedding)
2. Entraînement du modèle (RandomForest)
3. Sauvegarde des artifacts (model, scaler, PCA, label encoders)
4. Génération du fichier `ref_data.csv`

In [ ]:
import sys
import os

# Add scripts to path
sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))

import pandas as pd
import numpy as np
from model_utils import (
    load_raw_data, preprocess_features, encode_target,
    build_embedding, create_ref_data, train_model,
    save_artifact, load_artifact, FEATURE_COLS
)

import model_utils
model_utils.ARTIFACTS_DIR = os.path.join(os.getcwd(), "artifacts")
model_utils.DATA_DIR = os.path.join(os.getcwd(), "data")

print("Modules loaded successfully")

## 1. Exploration des données

In [ ]:
# Load the dataset
raw_path = os.path.join(os.getcwd(), "mental_health_dataset.csv")
df = load_raw_data(raw_path)
print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:")
print(df["Mental_Health_Condition"].value_counts())
print(f"\nFirst few rows:")
df.head()

## 2. Création du fichier ref_data.csv (embedding PCA)

On applique le pipeline de transformation : LabelEncoding → StandardScaler → PCA (5 composantes).
Le fichier `ref_data.csv` contiendra les vecteurs PCA + la cible encodée.

In [ ]:
# Create ref_data.csv and save preprocessing artifacts
ref_data_path = os.path.join(model_utils.DATA_DIR, "ref_data.csv")
ref_df, label_encoders, scaler, pca = create_ref_data(raw_path, ref_data_path)

print(f"\nref_data.csv preview:")
ref_df.head()

## 3. Entraînement du modèle (RandomForest)

In [ ]:
# Train the model on the embedded data
pca_columns = [c for c in ref_df.columns if c.startswith("pca_")]
X = ref_df[pca_columns].values
y = ref_df["target"].values

model, metrics = train_model(X, y, save=True)

print(f"\nArtifacts saved in: {model_utils.ARTIFACTS_DIR}")
print(f"Files: {os.listdir(model_utils.ARTIFACTS_DIR)}")

## 4. Vérification du pipeline de prédiction

Test du pipeline complet : données brutes → encodage → scaling → PCA → prédiction

In [ ]:
from model_utils import transform_single_input

# Test with a sample input
sample_input = {
    "Age": 35,
    "Gender": "Male",
    "Occupation": "IT",
    "Country": "USA",
    "Severity": "Medium",
    "Consultation_History": "Yes",
    "Stress_Level": "High",
    "Sleep_Hours": 5.5,
    "Work_Hours": 55.0,
    "Physical_Activity_Hours": 2
}

# Load artifacts
model_loaded = load_artifact("model.pkl")
scaler_loaded = load_artifact("scaler.pkl")
pca_loaded = load_artifact("pca.pkl")
le_loaded = load_artifact("label_encoders.pkl")

# Transform and predict
X_embedded = transform_single_input(sample_input, le_loaded, scaler_loaded, pca_loaded)
prediction = model_loaded.predict(X_embedded)[0]
proba = model_loaded.predict_proba(X_embedded)[0]

print(f"Input: {sample_input}")
print(f"Embedded vector: {X_embedded[0]}")
print(f"Prediction: {'Yes' if prediction == 1 else 'No'}")
print(f"Probabilities: No={proba[0]:.4f}, Yes={proba[1]:.4f}")